# Prepare exact career-age composition

## Purpose

This notebook prepares the exact career-age composition used by the manuscript
analysis.

It starts from:

```text
data/disruption_analysis.csv
```
and reconstructs each focal paper's author composition at:
- publication debut, career age 0;
- exact career ages 1 through 10;
- career age greater than 10.
For each author, publication debut is defined as the year of the author's first
observed publication in SciSciNet-v2.

## Reconstruction
For every focal paper in the manuscript sample, the notebook:
1. identifies authors using `sciscinet_authors_paperid.parquet`;
2. reconstructs each target author's complete observed publication history;
3. obtains publication years from `sciscinet_papers.parquet`;
4. defines each author's debut year as the earliest observed publication year;
5. calculates exact career age on the focal paper;
6. aggregates exact career-age counts and shares at the paper level.
The SciSciNet-v2 author-paper mapping provides complete author linkage for
29,054,201 of the 29,054,261 focal papers. The remaining 60 papers are retained
in the handoff but are excluded from exact-career-age analyses by the manuscript
analysis notebook.

## Main output
```text
data/sciscinet_exact_career_age_v2/
    v9_exact_career_age_handoff.parquet
```
The notebook also retains intermediate author-history files used by the
one-publication-debutant robustness analysis in the manuscript notebook.

## 0. Dependencies and SciSciNet-v2 access

This notebook uses:

- Polars for the manuscript analysis CSV and Parquet processing;
- DuckDB for the large author-history joins;
- Hugging Face Hub for SciSciNet-v2 source files.

Run `01_prepare_sciscinet_fields.ipynb` first. When available, this notebook
reuses the exact SciSciNet-v2 revision recorded by that notebook.

The notebook downloads and caches:

- `sciscinet_authors_paperid.parquet`
- `sciscinet_papers.parquet`

under:

```text
data/sciscinet_v2_public/
```
Set HF_TOKEN to a Hugging Face read token or enter the token when prompted.

In [ ]:
%pip install -U polars pyarrow pandas numpy duckdb huggingface_hub hf_xet psutil

## 1. Configuration

In [ ]:
import os
os.environ.setdefault("POLARS_MAX_THREADS", "16")
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

from pathlib import Path
from time import perf_counter
from getpass import getpass
import json
import platform
import shutil
import sys

import duckdb
import numpy as np
import pandas as pd
import polars as pl
import psutil

from huggingface_hub import HfApi, hf_hub_download
from huggingface_hub.errors import GatedRepoError, HfHubHTTPError

# ------------------------------------------------------------------
# Main analysis data
# ------------------------------------------------------------------
DATA_PATH = Path("data/disruption_analysis.csv")

# Reuse the exact SciSciNet-v2 revision from notebook 01 when available.
PRIOR_FIELD_MANIFEST = Path(
    "data/sciscinet_field_reconstruction_v2/field_reconstruction_manifest.json"
)

# ------------------------------------------------------------------
# SciSciNet-v2 source files and outputs
# ------------------------------------------------------------------
SOURCE_DIR = Path("data/sciscinet_v2_public")
WORK_DIR = Path("data/sciscinet_exact_career_age_v2")

SOURCE_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

REPO_ID = "Northwestern-CSSI/sciscinet-v2"

AUTHOR_PAPER_FILENAME = "sciscinet_authors_paperid.parquet"
PAPERS_FILENAME = "sciscinet_papers.parquet"

AUTHOR_PAPER_PATH = SOURCE_DIR / AUTHOR_PAPER_FILENAME
PAPERS_PATH = SOURCE_DIR / PAPERS_FILENAME

TARGET_PAPERS = WORK_DIR / "01_target_papers_v9_sample.parquet"
TARGET_LINKS = WORK_DIR / "02_target_paper_author_links.parquet"
TARGET_AUTHORS = WORK_DIR / "03_target_authors.parquet"
TARGET_AUTHOR_DEBUT = WORK_DIR / "04_target_author_debut_year.parquet"
EXACT_AGE_SIDECAR = WORK_DIR / "05_exact_career_age_sidecar.parquet"
V9_HANDOFF = WORK_DIR / "v9_exact_career_age_handoff.parquet"

VALIDATION_CSV = WORK_DIR / "validation_summary.csv"
MISMATCH_SAMPLE = WORK_DIR / "validation_mismatch_sample.parquet"
MANIFEST_JSON = WORK_DIR / "exact_career_age_manifest.json"

DUCKDB_TEMP_DIR = WORK_DIR / "duckdb_temp"
DUCKDB_TEMP_DIR.mkdir(parents=True, exist_ok=True)

DUCKDB_MEMORY_LIMIT = "40GB"
DUCKDB_THREADS = min(16, os.cpu_count() or 8)

START_YEAR = 1941
MAX_TEAM_SIZE = 40

# Manuscript analysis contract.
EXPECTED_MAIN_N = 29_054_261
EXPECTED_EXACT_N = 29_054_201
EXPECTED_INCOMPLETE_N = 60

RATIO_TOL = 1e-6
AVG_AGE_TOL = 1e-6

# Rebuild controls. Leave False for ordinary reruns.
FORCE_DOWNLOAD = False
FORCE_REBUILD_TARGETS = False
FORCE_REBUILD_DEBUT = False
FORCE_REBUILD_SIDECAR = False

# Hugging Face authentication:
# Prefer HF_TOKEN from the environment; otherwise prompt securely at runtime.
HF_TOKEN = os.environ.get("HF_TOKEN")

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Polars:", pl.__version__)
print("DuckDB:", duckdb.__version__)
print("Logical CPUs:", os.cpu_count())
print("DuckDB threads:", DUCKDB_THREADS)
print("RAM (GB):", round(psutil.virtual_memory().total / 2**30, 2))
print(
    "Free disk on working drive (GB):",
    round(shutil.disk_usage(WORK_DIR.resolve().anchor).free / 2**30, 1),
)
print()
print("Analysis CSV:", DATA_PATH.resolve())
print("SciSciNet source directory:", SOURCE_DIR.resolve())
print("Working directory:", WORK_DIR.resolve())

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find the analysis CSV:\n{DATA_PATH.resolve()}"
    )

## 2. Helper functions and SciSciNet-v2 revision

When the field-preparation manifest from notebook 01 is available, this
notebook uses the same resolved SciSciNet-v2 revision. This keeps the two
preparation stages tied to the same source snapshot.

If the manifest is unavailable, the notebook resolves the current `main`
revision and records the resulting commit SHA.

In [ ]:
def qident(name):
    """Quote a SQL identifier for DuckDB."""
    return '"' + str(name).replace('"', '""') + '"'


def qpath(path):
    """Absolute path using forward slashes, quoted safely for DuckDB SQL."""
    return str(Path(path).resolve()).replace("\\", "/").replace("'", "''")


def resolve_column(schema, aliases, semantic_name, required=True):
    names = list(schema.names())
    lower = {c.lower(): c for c in names}

    for alias in aliases:
        if alias.lower() in lower:
            return lower[alias.lower()]

    if required:
        raise ValueError(
            f"Could not resolve {semantic_name!r}.\n"
            f"Tried aliases: {aliases}\n"
            f"Available columns: {names}"
        )
    return None


def parquet_has_columns(path, required_columns):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        schema = pl.scan_parquet(path).collect_schema()
        return set(required_columns).issubset(set(schema.names()))
    except Exception:
        return False


def remove_if_invalid(path, required_columns):
    path = Path(path)
    if path.exists() and not parquet_has_columns(path, required_columns):
        print("Removing incomplete/incompatible file:", path.resolve())
        path.unlink(missing_ok=True)


def make_duckdb():
    con = duckdb.connect()
    con.execute(f"SET threads={DUCKDB_THREADS}")
    con.execute(f"SET memory_limit='{DUCKDB_MEMORY_LIMIT}'")
    con.execute(f"SET temp_directory='{qpath(DUCKDB_TEMP_DIR)}'")
    con.execute("SET preserve_insertion_order=false")
    return con


prior_revision = None

if PRIOR_FIELD_MANIFEST.exists():
    try:
        with open(PRIOR_FIELD_MANIFEST, "r", encoding="utf-8") as f:
            prior_manifest = json.load(f)
        prior_revision = prior_manifest.get("sciscinet_resolved_revision")
    except Exception as e:
        print("Could not read prior field manifest:", repr(e))

REQUESTED_REVISION = prior_revision or "main"

print("Requested SciSciNet-v2 revision:", REQUESTED_REVISION)
if prior_revision:
    print("This revision was recovered from the prior field-reconstruction manifest.")
else:
    print("No prior resolved revision found; the notebook will resolve 'main' once authenticated.")

## 3. Reconstruct the manuscript focal-paper sample

This cell reproduces the paper-level filtering used by the manuscript analysis:

- document type = `article`;
- publication year ≥ 1941;
- team size ≤ 40;
- removal of internally inconsistent all-debutant records.

The existing broad career-stage counts and shares are retained so that the
exact-age reconstruction can later be checked against them.

`target_row` is assigned after filtering and preserves source order. The exact
career-age handoff is written in this same order for direct alignment with the
manuscript analysis notebook.

In [ ]:
TARGET_REQUIRED = {
    "target_row",
    "paperid",
    "year",
    "team_size",
    "first_time_author_count",
    "early_career_author_count",
    "mid_career_author_count",
    "senior_author_count",
    "first_time_author_ratio",
    "early_career_author_ratio",
    "mid_career_author_ratio",
    "senior_author_ratio",
    "avg_career_age",
}

remove_if_invalid(TARGET_PAPERS, TARGET_REQUIRED)

if FORCE_REBUILD_TARGETS and TARGET_PAPERS.exists():
    TARGET_PAPERS.unlink()

if not TARGET_PAPERS.exists():
    t0 = perf_counter()

    raw_schema = pl.scan_csv(
        DATA_PATH,
        infer_schema_length=100_000,
        ignore_errors=False,
        try_parse_dates=False,
    ).collect_schema()

    needed = [
        "paperid",
        "doctype",
        "year",
        "team_size",
        "first_time_author_ratio",
        "early_career_author_ratio",
        "mid_career_author_ratio",
        "senior_author_ratio",
        "first_time_author_count",
        "early_career_author_count",
        "mid_career_author_count",
        "senior_author_count",
        "avg_career_age",
        "senior_author_avg_disruption",
        "mid_author_avg_disruption",
        "early_author_avg_disruption",
    ]

    missing = [c for c in needed if c not in raw_schema.names()]
    if missing:
        raise ValueError(
            "The manuscript CSV is missing columns required to reproduce "
            "the manuscript analysis sample:\n  - " + "\n  - ".join(missing)
        )

    def bad_all_beginner_expr():
        signals = [
            (pl.col("avg_career_age") > 0).fill_null(False),
            (pl.col("senior_author_avg_disruption") > 0).fill_null(False),
            (pl.col("senior_author_ratio") > 0).fill_null(False),
            (pl.col("mid_career_author_ratio") > 0).fill_null(False),
            (pl.col("mid_author_avg_disruption") > 0).fill_null(False),
            (pl.col("early_career_author_ratio") > 0).fill_null(False),
            (pl.col("early_author_avg_disruption") > 0).fill_null(False),
        ]
        return (
            (pl.col("first_time_author_ratio") == 1)
            & pl.any_horizontal(signals)
        ).fill_null(False)

    lf = (
        pl.scan_csv(
            DATA_PATH,
            infer_schema_length=100_000,
            ignore_errors=False,
            try_parse_dates=False,
        )
        .select(needed)
        .with_columns([
            pl.col("paperid").cast(pl.String),
            pl.col("year").cast(pl.Int32, strict=False),
            pl.col("team_size").cast(pl.Int16, strict=False),
        ])
        .filter(pl.col("doctype") == "article")
        .filter(pl.col("team_size") <= MAX_TEAM_SIZE)
        .filter(pl.col("year") >= START_YEAR)
        .filter(~bad_all_beginner_expr())
        .drop([
            "doctype",
            "senior_author_avg_disruption",
            "mid_author_avg_disruption",
            "early_author_avg_disruption",
        ])
        .with_row_index("target_row")
    )

    print("Streaming manuscript target sample to:")
    print(" ", TARGET_PAPERS.resolve())

    lf.sink_parquet(
        TARGET_PAPERS,
        compression="zstd",
        compression_level=3,
        statistics=True,
        maintain_order=True,
    )

    print(f"Finished in {(perf_counter() - t0) / 60:.2f} minutes.")
else:
    print("Reusing:", TARGET_PAPERS.resolve())


target_audit = (
    pl.scan_parquet(TARGET_PAPERS)
    .select([
        pl.len().alias("rows"),
        pl.col("paperid").n_unique().alias("unique_paperids"),
        pl.col("year").min().alias("min_year"),
        pl.col("year").max().alias("max_year"),
        pl.col("team_size").min().alias("min_team_size"),
        pl.col("team_size").max().alias("max_team_size"),
        (
            pl.col("first_time_author_count")
            + pl.col("early_career_author_count")
            + pl.col("mid_career_author_count")
            + pl.col("senior_author_count")
            != pl.col("team_size")
        ).sum().alias("existing_count_sum_ne_team_size"),
        (
            (
                pl.col("first_time_author_ratio")
                + pl.col("early_career_author_ratio")
                + pl.col("mid_career_author_ratio")
                + pl.col("senior_author_ratio")
                - 1.0
            ).abs() > RATIO_TOL
        ).sum().alias("existing_ratio_sum_not_one"),
    ])
    .collect()
)

display(target_audit.to_pandas())

n_target = int(target_audit["rows"][0])
n_unique_target = int(target_audit["unique_paperids"][0])

if n_target != n_unique_target:
    raise RuntimeError(
        f"Target paper IDs are not unique: {n_target:,} rows but "
        f"{n_unique_target:,} unique paper IDs."
    )

if n_target != EXPECTED_MAIN_N:
    raise RuntimeError(
        f"Target sample has {n_target:,} papers; expected {EXPECTED_MAIN_N:,}. "
        "The local CSV or filtering contract differs from the manuscript sample."
    )

print("PASS: target paper count and uniqueness match the manuscript sample.")

## 4. Authenticate to Hugging Face and pin the SciSciNet-v2 revision

The token is read from `HF_TOKEN` if set; otherwise the notebook prompts securely. It is **not written to disk or the manifest**.

In [ ]:
if not HF_TOKEN:
    HF_TOKEN = getpass(
        "Paste a Hugging Face READ token for the authorized account (input hidden): "
    ).strip()

if not HF_TOKEN:
    raise RuntimeError("No Hugging Face token supplied.")

api = HfApi(token=HF_TOKEN)

try:
    who = api.whoami()
    repo_info = api.dataset_info(
        REPO_ID,
        revision=REQUESTED_REVISION,
    )
except GatedRepoError as e:
    raise RuntimeError(
        "This Hugging Face account does not currently have SciSciNet-v2 access. "
        "Open the dataset page while signed into the same account, accept/request "
        "the access conditions, then rerun this cell."
    ) from e
except HfHubHTTPError as e:
    raise RuntimeError(
        "Hugging Face authentication or repository access failed."
    ) from e

hf_username = (
    who.get("name")
    or who.get("fullname")
    or who.get("email")
    or "<unknown>"
)

RESOLVED_REVISION = repo_info.sha

repo_files = set(api.list_repo_files(
    REPO_ID,
    repo_type="dataset",
    revision=RESOLVED_REVISION,
))

for required_file in [AUTHOR_PAPER_FILENAME, PAPERS_FILENAME]:
    if required_file not in repo_files:
        raise RuntimeError(
            f"{required_file} is not present at SciSciNet-v2 revision "
            f"{RESOLVED_REVISION}."
        )

print("Authenticated Hugging Face account:", hf_username)
print("SciSciNet-v2 access: PASS")
print("Requested revision:", REQUESTED_REVISION)
print("Resolved revision:", RESOLVED_REVISION)
print("Required source files are present.")

## 5. Download the two SciSciNet-v2 source files

The files are kept under `data\sciscinet_v2_public` and reused on later runs.

This notebook needs only:

- `sciscinet_authors_paperid.parquet` — author ↔ paper links;
- `sciscinet_papers.parquet` — publication years.

No citation, reference, field, embedding, or paper-details files are required.

In [ ]:
def valid_parquet_metadata(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        pl.scan_parquet(path).collect_schema()
        return True
    except Exception:
        return False


def get_sciscinet_file(filename, expected_path, force=False):
    expected_path = Path(expected_path)

    if expected_path.exists() and not force:
        if valid_parquet_metadata(expected_path):
            print(
                f"Reusing {filename}: "
                f"{expected_path.stat().st_size / 2**30:.2f} GB"
            )
            return expected_path
        else:
            print("Removing invalid/incomplete local file:", expected_path.resolve())
            expected_path.unlink(missing_ok=True)

    print("Downloading:", filename)
    t0 = perf_counter()

    downloaded = hf_hub_download(
        repo_id=REPO_ID,
        filename=filename,
        repo_type="dataset",
        revision=RESOLVED_REVISION,
        local_dir=SOURCE_DIR,
        force_download=force,
        token=HF_TOKEN,
    )

    downloaded = Path(downloaded)

    if not valid_parquet_metadata(downloaded):
        raise RuntimeError(
            f"Downloaded file does not appear to be a readable Parquet file: "
            f"{downloaded}"
        )

    print(
        "Ready:",
        downloaded.resolve(),
        f"({downloaded.stat().st_size / 2**30:.2f} GB, "
        f"{(perf_counter() - t0) / 60:.2f} min)",
    )
    return downloaded


AUTHOR_PAPER_PATH = get_sciscinet_file(
    AUTHOR_PAPER_FILENAME,
    AUTHOR_PAPER_PATH,
    FORCE_DOWNLOAD,
)

PAPERS_PATH = get_sciscinet_file(
    PAPERS_FILENAME,
    PAPERS_PATH,
    FORCE_DOWNLOAD,
)

print()
print("Total downloaded source size (GB):",
      round((AUTHOR_PAPER_PATH.stat().st_size + PAPERS_PATH.stat().st_size) / 2**30, 2))

## 6. Inspect and resolve the public v2 schemas

This is an explicit guard against silently assuming column names. The notebook supports either a scalar `paperid` column or a list-valued `paperids` column in the author–paper file.

In [ ]:
author_paper_schema = pl.scan_parquet(AUTHOR_PAPER_PATH).collect_schema()
papers_schema = pl.scan_parquet(PAPERS_PATH).collect_schema()

print("=== sciscinet_authors_paperid.parquet ===")
for name, dtype in author_paper_schema.items():
    print(f"{name:40s} {dtype}")

print("\n=== sciscinet_papers.parquet ===")
for name, dtype in papers_schema.items():
    print(f"{name:40s} {dtype}")


AP_AUTHOR_ID = resolve_column(
    author_paper_schema,
    ["authorid", "author_id", "id"],
    "author ID in author-paper mapping",
)

AP_PAPER_ID = resolve_column(
    author_paper_schema,
    ["paperid", "paper_id", "workid", "work_id"],
    "paper ID in author-paper mapping",
    required=False,
)

AP_PAPER_IDS = resolve_column(
    author_paper_schema,
    ["paperids", "paper_ids", "workids", "work_ids"],
    "paper-ID list in author-paper mapping",
    required=False,
)

if AP_PAPER_ID is None and AP_PAPER_IDS is None:
    raise ValueError(
        "Could not find either a scalar paper-ID column or a paper-ID-list "
        "column in sciscinet_authors_paperid.parquet."
    )

P_PAPER_ID = resolve_column(
    papers_schema,
    ["paperid", "paper_id", "workid", "work_id"],
    "paper ID in papers",
)

P_YEAR = resolve_column(
    papers_schema,
    ["year", "publication_year", "pub_year"],
    "publication year in papers",
)

print("\nResolved columns:")
print(" author-paper author ID ->", AP_AUTHOR_ID)
print(" author-paper paper ID  ->", AP_PAPER_ID)
print(" author-paper paper list->", AP_PAPER_IDS)
print(" papers paper ID        ->", P_PAPER_ID)
print(" papers year            ->", P_YEAR)


if AP_PAPER_ID is not None:
    AUTHOR_PAPER_RELATION_SQL = f"""
        SELECT
            CAST({qident(AP_AUTHOR_ID)} AS VARCHAR) AS authorid,
            CAST({qident(AP_PAPER_ID)} AS VARCHAR) AS paperid
        FROM read_parquet('{qpath(AUTHOR_PAPER_PATH)}')
        WHERE {qident(AP_AUTHOR_ID)} IS NOT NULL
          AND {qident(AP_PAPER_ID)} IS NOT NULL
    """
else:
    AUTHOR_PAPER_RELATION_SQL = f"""
        SELECT
            CAST({qident(AP_AUTHOR_ID)} AS VARCHAR) AS authorid,
            CAST(UNNEST({qident(AP_PAPER_IDS)}) AS VARCHAR) AS paperid
        FROM read_parquet('{qpath(AUTHOR_PAPER_PATH)}')
        WHERE {qident(AP_AUTHOR_ID)} IS NOT NULL
          AND {qident(AP_PAPER_IDS)} IS NOT NULL
    """


PAPERS_RELATION_SQL = f"""
    SELECT
        CAST({qident(P_PAPER_ID)} AS VARCHAR) AS paperid,
        TRY_CAST({qident(P_YEAR)} AS INTEGER) AS year
    FROM read_parquet('{qpath(PAPERS_PATH)}')
    WHERE {qident(P_PAPER_ID)} IS NOT NULL
"""

## 7. Restrict the full author–paper graph to authors on the 29M focal papers

This is the first large local join. DuckDB scans the author–paper Parquet and semijoins it to the target paper IDs.

The output contains one distinct `(target_row, paperid, authorid)` record per focal-paper author.

In [ ]:
remove_if_invalid(
    TARGET_LINKS,
    {"target_row", "paperid", "authorid"},
)

if FORCE_REBUILD_TARGETS and TARGET_LINKS.exists():
    TARGET_LINKS.unlink()

if not TARGET_LINKS.exists():
    t0 = perf_counter()
    con = make_duckdb()

    sql = f"""
    COPY (
        SELECT DISTINCT
            t.target_row,
            t.paperid,
            ap.authorid
        FROM read_parquet('{qpath(TARGET_PAPERS)}') AS t
        INNER JOIN (
            {AUTHOR_PAPER_RELATION_SQL}
        ) AS ap
        ON ap.paperid = t.paperid
    )
    TO '{qpath(TARGET_LINKS)}'
    (FORMAT PARQUET, COMPRESSION ZSTD);
    """

    print("Building focal-paper author links...")
    con.execute(sql)
    con.close()
    print(f"Finished in {(perf_counter() - t0) / 60:.2f} minutes.")
else:
    print("Reusing:", TARGET_LINKS.resolve())


con = make_duckdb()
linkage_audit = con.execute(f"""
WITH linked AS (
    SELECT
        target_row,
        COUNT(*) AS n_linked_authors
    FROM read_parquet('{qpath(TARGET_LINKS)}')
    GROUP BY target_row
)
SELECT
    COUNT(*) AS n_target_papers,
    SUM(CASE WHEN l.target_row IS NULL THEN 1 ELSE 0 END) AS papers_with_no_author_link,
    SUM(CASE WHEN COALESCE(l.n_linked_authors, 0) = t.team_size THEN 1 ELSE 0 END)
        AS papers_matching_team_size,
    SUM(ABS(COALESCE(l.n_linked_authors, 0) - t.team_size))
        AS total_abs_team_size_difference,
    MAX(ABS(COALESCE(l.n_linked_authors, 0) - t.team_size))
        AS max_abs_team_size_difference,
    SUM(t.team_size) AS expected_author_slots,
    SUM(COALESCE(l.n_linked_authors, 0)) AS linked_author_slots
FROM read_parquet('{qpath(TARGET_PAPERS)}') AS t
LEFT JOIN linked AS l USING (target_row)
""").fetchdf()
con.close()

display(linkage_audit)

if int(linkage_audit["papers_with_no_author_link"].iloc[0]) > 0:
    print(
        "WARNING: Some focal papers have no author links in the current "
        "SciSciNet-v2 author-paper mapping. The final validation will determine "
        "whether this prevents faithful reconstruction."
    )

## 8. Cache the target-author set

This is small relative to the full author–paper graph and is reused in the career-start query.

In [ ]:
remove_if_invalid(TARGET_AUTHORS, {"authorid"})

if FORCE_REBUILD_TARGETS and TARGET_AUTHORS.exists():
    TARGET_AUTHORS.unlink()

if not TARGET_AUTHORS.exists():
    con = make_duckdb()
    con.execute(f"""
    COPY (
        SELECT DISTINCT authorid
        FROM read_parquet('{qpath(TARGET_LINKS)}')
        WHERE authorid IS NOT NULL
    )
    TO '{qpath(TARGET_AUTHORS)}'
    (FORMAT PARQUET, COMPRESSION ZSTD);
    """)
    con.close()

target_author_audit = (
    pl.scan_parquet(TARGET_AUTHORS)
    .select(pl.len().alias("unique_target_authors"))
    .collect()
)

display(target_author_audit.to_pandas())

## 9. Derive first observed publication year for every target author

This is the heaviest step.

For every author who appears on at least one focal paper, the notebook scans that author's complete SciSciNet-v2 publication history and takes:

\[
\mathrm{DebutYear}_a = \min_{p \in \mathcal{P}_a} \mathrm{Year}_p.
\]

This reproduces the conceptual definition needed by the manuscript: career age is based on the author's first observed publication in SciSciNet, not merely the first paper inside the filtered 1941–2020 article sample.

In [ ]:
remove_if_invalid(
    TARGET_AUTHOR_DEBUT,
    {"authorid", "debut_year"},
)

if FORCE_REBUILD_DEBUT and TARGET_AUTHOR_DEBUT.exists():
    TARGET_AUTHOR_DEBUT.unlink()

if not TARGET_AUTHOR_DEBUT.exists():
    t0 = perf_counter()
    con = make_duckdb()

    sql = f"""
    COPY (
        SELECT
            ap.authorid,
            MIN(p.year) AS debut_year
        FROM (
            {AUTHOR_PAPER_RELATION_SQL}
        ) AS ap
        INNER JOIN read_parquet('{qpath(TARGET_AUTHORS)}') AS ta
            ON ta.authorid = ap.authorid
        INNER JOIN (
            {PAPERS_RELATION_SQL}
        ) AS p
            ON p.paperid = ap.paperid
        WHERE p.year IS NOT NULL
        GROUP BY ap.authorid
    )
    TO '{qpath(TARGET_AUTHOR_DEBUT)}'
    (FORMAT PARQUET, COMPRESSION ZSTD);
    """

    print("Deriving debut years for target authors...")
    print("This step scans the full author-paper mapping and paper-year table.")
    con.execute(sql)
    con.close()

    print(f"Finished in {(perf_counter() - t0) / 60:.2f} minutes.")
else:
    print("Reusing:", TARGET_AUTHOR_DEBUT.resolve())


con = make_duckdb()
debut_audit = con.execute(f"""
SELECT
    (SELECT COUNT(*) FROM read_parquet('{qpath(TARGET_AUTHORS)}'))
        AS target_authors,
    (SELECT COUNT(*) FROM read_parquet('{qpath(TARGET_AUTHOR_DEBUT)}'))
        AS authors_with_debut_year,
    (
        SELECT COUNT(*)
        FROM read_parquet('{qpath(TARGET_AUTHORS)}') ta
        LEFT JOIN read_parquet('{qpath(TARGET_AUTHOR_DEBUT)}') d USING (authorid)
        WHERE d.authorid IS NULL
    ) AS authors_missing_debut_year,
    (SELECT MIN(debut_year) FROM read_parquet('{qpath(TARGET_AUTHOR_DEBUT)}'))
        AS min_debut_year,
    (SELECT MAX(debut_year) FROM read_parquet('{qpath(TARGET_AUTHOR_DEBUT)}'))
        AS max_debut_year
""").fetchdf()
con.close()

display(debut_audit)

## 10. Build exact career-age counts and shares for every focal paper

For each focal paper, the notebook computes the exact author career ages and then saves:

- `career_age_0_count`, ..., `career_age_10_count`, `career_age_11plus_count`;
- corresponding `_share` columns;
- linked-author and missing-debut diagnostics;
- reconstructed mean career age.

The shares use the existing manuscript `team_size` as the denominator, which makes the new quantities directly comparable with the original broad career-stage ratios.

In [ ]:
required_sidecar_cols = {
    "target_row",
    "paperid",
    "year",
    "team_size",
    "career_age_0_count",
    "career_age_10_count",
    "career_age_11plus_count",
    "career_age_0_share",
    "career_age_10_share",
    "career_age_11plus_share",
    "n_linked_authors",
    "missing_debut_year_count",
    "negative_career_age_count",
    "reconstructed_avg_career_age",
}

remove_if_invalid(EXACT_AGE_SIDECAR, required_sidecar_cols)

if FORCE_REBUILD_SIDECAR and EXACT_AGE_SIDECAR.exists():
    EXACT_AGE_SIDECAR.unlink()

age_count_sql = ",\n".join(
    [
        f"COUNT(*) FILTER (WHERE career_age = {age})::SMALLINT "
        f"AS career_age_{age}_count"
        for age in range(11)
    ]
)

age_share_sql = ",\n".join(
    [
        f"(a.career_age_{age}_count::DOUBLE / NULLIF(t.team_size, 0)) "
        f"AS career_age_{age}_share"
        for age in range(11)
    ]
)

if not EXACT_AGE_SIDECAR.exists():
    t0 = perf_counter()
    con = make_duckdb()

    exact_count_select = ", ".join(
        [
            f"COALESCE(a.career_age_{age}_count, 0)::SMALLINT "
            f"AS career_age_{age}_count"
            for age in range(11)
        ]
    )

    sql = f"""
    COPY (
        WITH author_ages AS (
            SELECT
                l.target_row,
                l.paperid,
                l.authorid,
                t.year,
                d.debut_year,
                CASE
                    WHEN d.debut_year IS NULL THEN NULL
                    ELSE t.year - d.debut_year
                END AS career_age
            FROM read_parquet('{qpath(TARGET_LINKS)}') AS l
            INNER JOIN read_parquet('{qpath(TARGET_PAPERS)}') AS t
                USING (target_row, paperid)
            LEFT JOIN read_parquet('{qpath(TARGET_AUTHOR_DEBUT)}') AS d
                USING (authorid)
        ),
        age_agg AS (
            SELECT
                target_row,
                paperid,
                COUNT(*)::SMALLINT AS n_linked_authors,
                COUNT(*) FILTER (WHERE debut_year IS NULL)::SMALLINT
                    AS missing_debut_year_count,
                COUNT(*) FILTER (WHERE career_age < 0)::SMALLINT
                    AS negative_career_age_count,
                AVG(career_age) FILTER (WHERE career_age >= 0)
                    AS reconstructed_avg_career_age,
                {age_count_sql},
                COUNT(*) FILTER (WHERE career_age >= 11)::SMALLINT
                    AS career_age_11plus_count
            FROM author_ages
            GROUP BY target_row, paperid
        )
        SELECT
            t.*,
            COALESCE(a.n_linked_authors, 0)::SMALLINT AS n_linked_authors,
            COALESCE(a.missing_debut_year_count, 0)::SMALLINT
                AS missing_debut_year_count,
            COALESCE(a.negative_career_age_count, 0)::SMALLINT
                AS negative_career_age_count,
            a.reconstructed_avg_career_age,
            {exact_count_select},
            COALESCE(a.career_age_11plus_count, 0)::SMALLINT
                AS career_age_11plus_count,
            {age_share_sql},
            (
                COALESCE(a.career_age_11plus_count, 0)::DOUBLE
                / NULLIF(t.team_size, 0)
            ) AS career_age_11plus_share
        FROM read_parquet('{qpath(TARGET_PAPERS)}') AS t
        LEFT JOIN age_agg AS a
            USING (target_row, paperid)
        ORDER BY t.target_row
    )
    TO '{qpath(EXACT_AGE_SIDECAR)}'
    (FORMAT PARQUET, COMPRESSION ZSTD);
    """

    print("Building exact-career-age paper sidecar...")
    con.execute(sql)
    con.close()

    print(f"Finished in {(perf_counter() - t0) / 60:.2f} minutes.")
else:
    print("Reusing:", EXACT_AGE_SIDECAR.resolve())

## 11. Validate the manuscript exact-career-age reconstruction

The SciSciNet-v2 author-paper mapping links all expected authors for
29,054,201 of the 29,054,261 focal papers.

For 60 focal papers, the author-paper mapping contains one fewer linked author
than the team size recorded in the analysis dataset. These 60 papers are kept
in the handoff so that row alignment with the full manuscript sample is
preserved, but the manuscript notebook excludes them from exact-career-age
analyses.

Validation therefore requires:

- 29,054,261 focal papers in the full handoff;
- 29,054,201 papers with complete exact-age reconstruction;
- exactly 60 incomplete-linkage papers;
- exactly one missing linked author on each incomplete paper;
- no missing debut year among linked authors;
- no negative career ages;
- exact agreement between reconstructed and existing broad career-stage
  counts and shares for every complete paper;
- exact agreement in reconstructed mean career age for every complete paper.

The 60 incomplete-linkage papers are saved separately for transparency.

In [ ]:
s = pl.scan_parquet(EXACT_AGE_SIDECAR).with_columns([
    sum(
        [pl.col(f"career_age_{a}_count") for a in range(1, 6)]
    ).alias("reconstructed_1_5_count"),

    sum(
        [pl.col(f"career_age_{a}_count") for a in range(6, 11)]
    ).alias("reconstructed_6_10_count"),

    (
        sum([pl.col(f"career_age_{a}_count") for a in range(11)])
        + pl.col("career_age_11plus_count")
    ).alias("reconstructed_total_count"),

    (
        sum([pl.col(f"career_age_{a}_share") for a in range(11)])
        + pl.col("career_age_11plus_share")
    ).alias("reconstructed_share_sum"),

    sum(
        [pl.col(f"career_age_{a}_share") for a in range(1, 6)]
    ).alias("reconstructed_1_5_share"),

    sum(
        [pl.col(f"career_age_{a}_share") for a in range(6, 11)]
    ).alias("reconstructed_6_10_share"),
])

complete_exact_expr = (
    (pl.col("n_linked_authors") == pl.col("team_size"))
    & (pl.col("missing_debut_year_count") == 0)
    & (pl.col("negative_career_age_count") == 0)
    & (pl.col("reconstructed_total_count") == pl.col("team_size"))
)

overall = (
    s.select([
        pl.len().alias("n_papers"),
        pl.col("paperid").n_unique().alias("unique_paperids"),
        pl.col("target_row").min().alias("target_row_min"),
        pl.col("target_row").max().alias("target_row_max"),
        complete_exact_expr.sum().alias("papers_complete_exact_age"),
        (~complete_exact_expr).sum().alias("papers_incomplete_exact_age"),
        (
            pl.col("team_size") - pl.col("n_linked_authors")
        ).sum().alias("missing_author_slots"),
        (
            pl.col("team_size") - pl.col("n_linked_authors")
        ).max().alias("max_missing_author_slots"),
        pl.col("missing_debut_year_count")
        .sum().alias("author_slots_missing_debut_year"),
        pl.col("negative_career_age_count")
        .sum().alias("author_slots_negative_career_age"),
    ])
    .collect()
    .row(0, named=True)
)

complete = s.filter(complete_exact_expr)

complete_checks = (
    complete.select([
        (
            (pl.col("reconstructed_share_sum") - 1.0).abs() > RATIO_TOL
        ).sum().alias("complete_papers_share_sum_not_one"),

        (
            pl.col("career_age_0_count") != pl.col("first_time_author_count")
        ).sum().alias("complete_papers_age0_count_mismatch"),

        (
            pl.col("reconstructed_1_5_count")
            != pl.col("early_career_author_count")
        ).sum().alias("complete_papers_age1_5_count_mismatch"),

        (
            pl.col("reconstructed_6_10_count")
            != pl.col("mid_career_author_count")
        ).sum().alias("complete_papers_age6_10_count_mismatch"),

        (
            pl.col("career_age_11plus_count")
            != pl.col("senior_author_count")
        ).sum().alias("complete_papers_age11plus_count_mismatch"),

        (
            (
                pl.col("career_age_0_share")
                - pl.col("first_time_author_ratio")
            ).abs() > RATIO_TOL
        ).sum().alias("complete_papers_age0_share_mismatch"),

        (
            (
                pl.col("reconstructed_1_5_share")
                - pl.col("early_career_author_ratio")
            ).abs() > RATIO_TOL
        ).sum().alias("complete_papers_age1_5_share_mismatch"),

        (
            (
                pl.col("reconstructed_6_10_share")
                - pl.col("mid_career_author_ratio")
            ).abs() > RATIO_TOL
        ).sum().alias("complete_papers_age6_10_share_mismatch"),

        (
            (
                pl.col("career_age_11plus_share")
                - pl.col("senior_author_ratio")
            ).abs() > RATIO_TOL
        ).sum().alias("complete_papers_age11plus_share_mismatch"),

        (
            (
                pl.col("reconstructed_avg_career_age")
                - pl.col("avg_career_age")
            ).abs() > AVG_AGE_TOL
        ).fill_null(True).sum().alias(
            "complete_papers_avg_career_age_mismatch"
        ),

        (
            pl.col("reconstructed_avg_career_age")
            - pl.col("avg_career_age")
        ).abs().max().alias(
            "complete_max_abs_avg_career_age_difference"
        ),
    ])
    .collect()
    .row(0, named=True)
)

validation_record_raw = {
    **overall,
    **complete_checks,
}

validation_df = pd.DataFrame([validation_record_raw])
display(validation_df)

validation_df.to_csv(VALIDATION_CSV, index=False)
print("Saved validation summary:", VALIDATION_CSV.resolve())

# Save the known incomplete-linkage cases for transparency.
incomplete_sample_df = (
    s.filter(~complete_exact_expr)
    .collect()
)

if incomplete_sample_df.height > 0:
    incomplete_sample_df.write_parquet(
        MISMATCH_SAMPLE,
        compression="zstd",
    )
    print(
        f"Saved {incomplete_sample_df.height:,} incomplete-linkage papers to:",
        MISMATCH_SAMPLE.resolve(),
    )
else:
    MISMATCH_SAMPLE.unlink(missing_ok=True)

failed = {}

expected_equalities = {
    "n_papers": EXPECTED_MAIN_N,
    "unique_paperids": EXPECTED_MAIN_N,
    "papers_complete_exact_age": EXPECTED_EXACT_N,
    "papers_incomplete_exact_age": EXPECTED_INCOMPLETE_N,
    "missing_author_slots": EXPECTED_INCOMPLETE_N,
    "max_missing_author_slots": 1,
    "author_slots_missing_debut_year": 0,
    "author_slots_negative_career_age": 0,
}

for key, expected in expected_equalities.items():
    observed = int(validation_record_raw[key])
    if observed != expected:
        failed[key] = {
            "observed": observed,
            "expected": expected,
        }

complete_zero_checks = [
    "complete_papers_share_sum_not_one",
    "complete_papers_age0_count_mismatch",
    "complete_papers_age1_5_count_mismatch",
    "complete_papers_age6_10_count_mismatch",
    "complete_papers_age11plus_count_mismatch",
    "complete_papers_age0_share_mismatch",
    "complete_papers_age1_5_share_mismatch",
    "complete_papers_age6_10_share_mismatch",
    "complete_papers_age11plus_share_mismatch",
    "complete_papers_avg_career_age_mismatch",
]

for key in complete_zero_checks:
    observed = int(validation_record_raw[key])
    if observed != 0:
        failed[key] = {
            "observed": observed,
            "expected": 0,
        }

if failed:
    print("\nVALIDATION FAILED")
    print(json.dumps(failed, indent=2))
    raise RuntimeError(
        "Exact-career-age reconstruction does not match the manuscript "
        "analysis contract. Inspect validation_summary.csv and "
        "validation_mismatch_sample.parquet."
    )

print()
print(
    "PASS: exact-career-age reconstruction matches the "
    "manuscript analysis contract."
)
print(f"  Full focal-paper sample:       {EXPECTED_MAIN_N:,}")
print(f"  Complete exact-age papers:     {EXPECTED_EXACT_N:,}")
print(f"  Incomplete-linkage papers:     {EXPECTED_INCOMPLETE_N:,}")
print(
    "  Complete papers reproduce the broad career-stage "
    "variables exactly."
)

## 12. Create the exact-career-age handoff

The handoff contains one row for every focal paper and is ordered by
`target_row`, preserving the row order of the manuscript analysis sample.

The 60 known incomplete-linkage papers remain in the handoff. The manuscript
analysis notebook identifies them from the exact-age count totals and excludes
them only from analyses requiring exact career age.

The file also retains the existing broad career-stage variables so that row
alignment and career-stage reconciliation can be checked again when the
manuscript notebook loads it.

In [ ]:
handoff_cols = (
    [
        "target_row",
        "paperid",
        "year",
        "team_size",
        "first_time_author_count",
        "early_career_author_count",
        "mid_career_author_count",
        "senior_author_count",
        "first_time_author_ratio",
        "early_career_author_ratio",
        "mid_career_author_ratio",
        "senior_author_ratio",
        "avg_career_age",
        "reconstructed_avg_career_age",
    ]
    + [f"career_age_{a}_count" for a in range(11)]
    + ["career_age_11plus_count"]
    + [f"career_age_{a}_share" for a in range(11)]
    + ["career_age_11plus_share"]
)

if V9_HANDOFF.exists():
    V9_HANDOFF.unlink()

(
    pl.scan_parquet(EXACT_AGE_SIDECAR)
    .select(handoff_cols)
    .sort("target_row")
    .sink_parquet(
        V9_HANDOFF,
        compression="zstd",
        compression_level=3,
        statistics=True,
        maintain_order=True,
    )
)

handoff_audit = (
    pl.scan_parquet(V9_HANDOFF)
    .select([
        pl.len().alias("rows"),
        pl.col("target_row").min().alias("target_row_min"),
        pl.col("target_row").max().alias("target_row_max"),
        pl.col("paperid").n_unique().alias("unique_paperids"),
    ])
    .collect()
)

display(handoff_audit.to_pandas())

print("v9 handoff:", V9_HANDOFF.resolve())
print("Handoff size (GB):", round(V9_HANDOFF.stat().st_size / 2**30, 3))

## 13. Save provenance manifest and final audit

The manifest records the SciSciNet-v2 revision, source files, resolved schemas,
processing settings, validation results, and output paths used to prepare the
exact-career-age handoff.

In [ ]:
validation_record = {
    k: (
        None if pd.isna(vv)
        else float(vv) if isinstance(vv, (float, np.floating))
        else int(vv) if isinstance(vv, (int, np.integer))
        else vv
    )
    for k, vv in validation_df.iloc[0].to_dict().items()
}

manifest = {
    "purpose": (
        "Prepare exact author career ages 0-10 and 11+ for the "
        "manuscript analysis notebook."
    ),
    "analysis_input_csv": str(DATA_PATH.resolve()),
    "expected_main_n": EXPECTED_MAIN_N,
    "expected_exact_age_n": EXPECTED_EXACT_N,
    "expected_incomplete_linkage_n": EXPECTED_INCOMPLETE_N,
    "sample_filters": {
        "doctype": "article",
        "minimum_year": START_YEAR,
        "maximum_team_size": MAX_TEAM_SIZE,
        "bad_all_debutant_filter": (
            "same filtering expression used in the manuscript "
            "analysis notebook"
        ),
    },
    "sciscinet": {
        "repo_id": REPO_ID,
        "requested_revision": REQUESTED_REVISION,
        "resolved_revision": RESOLVED_REVISION,
        "huggingface_authenticated_account": hf_username,
        "huggingface_token_persisted": False,
    },
    "source_files": {
        "author_paper": {
            "path": str(AUTHOR_PAPER_PATH.resolve()),
            "filename": AUTHOR_PAPER_FILENAME,
            "size_bytes": AUTHOR_PAPER_PATH.stat().st_size,
        },
        "papers": {
            "path": str(PAPERS_PATH.resolve()),
            "filename": PAPERS_FILENAME,
            "size_bytes": PAPERS_PATH.stat().st_size,
        },
    },
    "resolved_schema": {
        "author_paper_authorid": AP_AUTHOR_ID,
        "author_paper_paperid": AP_PAPER_ID,
        "author_paper_paperids_list": AP_PAPER_IDS,
        "papers_paperid": P_PAPER_ID,
        "papers_year": P_YEAR,
    },
    "career_age_definition": (
        "debut_year(author) = minimum non-null publication year across "
        "the author's complete SciSciNet-v2 author-paper history; "
        "career_age = focal_paper_year - debut_year"
    ),
    "engines": {
        "csv_filtering": "Polars lazy streaming",
        "large_joins": "DuckDB",
        "duckdb_memory_limit": DUCKDB_MEMORY_LIMIT,
        "duckdb_threads": DUCKDB_THREADS,
        "duckdb_temp_directory": str(DUCKDB_TEMP_DIR.resolve()),
    },
    "validation": validation_record,
    "outputs": {
        "target_papers": str(TARGET_PAPERS.resolve()),
        "target_links": str(TARGET_LINKS.resolve()),
        "target_authors": str(TARGET_AUTHORS.resolve()),
        "target_author_debut": str(TARGET_AUTHOR_DEBUT.resolve()),
        "exact_age_sidecar": str(EXACT_AGE_SIDECAR.resolve()),
        "manuscript_handoff": str(V9_HANDOFF.resolve()),
        "validation_summary": str(VALIDATION_CSV.resolve()),
        "incomplete_linkage_sample": (
            str(MISMATCH_SAMPLE.resolve())
            if MISMATCH_SAMPLE.exists()
            else None
        ),
    },
}

with open(MANIFEST_JSON, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("=" * 88)
print("EXACT CAREER-AGE PREPARATION COMPLETE")
print("=" * 88)
print("SciSciNet-v2 revision:", RESOLVED_REVISION)
print("Full focal-paper sample:", f"{EXPECTED_MAIN_N:,}")
print("Complete exact-age papers:", f"{EXPECTED_EXACT_N:,}")
print("Known incomplete-linkage papers:", f"{EXPECTED_INCOMPLETE_N:,}")
print()
print("Primary handoff:")
print(" ", V9_HANDOFF.resolve())
print()
print("Validation summary:")
print(" ", VALIDATION_CSV.resolve())
print()
print("Manifest:")
print(" ", MANIFEST_JSON.resolve())

for p in [
    AUTHOR_PAPER_PATH,
    PAPERS_PATH,
    TARGET_PAPERS,
    TARGET_LINKS,
    TARGET_AUTHORS,
    TARGET_AUTHOR_DEBUT,
    EXACT_AGE_SIDECAR,
    V9_HANDOFF,
]:
    if Path(p).exists():
        print(
            f"{Path(p).name:45s} "
            f"{Path(p).stat().st_size / 2**30:8.3f} GB"
        )